# BERT-base-uncased — baseline puis tuning léger

Ce notebook est consacré au modèle **`bert-base-uncased`**. Il contient deux grandes parties. La première entraîne une version **non tunée** du modèle avec un jeu d’hyperparamètres de départ, afin d’obtenir une baseline transformer claire. La seconde lance une **phase de tuning léger** sur plusieurs configurations raisonnables, choisit la meilleure sur la validation, puis réentraîne cette meilleure configuration sur l’ensemble du train avant de l’évaluer sur le test.

Ces modèles sont bien des modèles de **deep learning**, plus précisément des **transformers préentraînés** fine-tunés pour une tâche de classification binaire. Le tuning est utile, mais il doit rester léger et pragmatique, car le coût de calcul augmente vite. L’idée ici est donc de tester quelques configurations pertinentes, pas d’explorer un espace gigantesque.


## Installation éventuelle

Décommente la cellule suivante si nécessaire.


In [2]:
pip install -q pandas numpy scikit-learn torch transformers datasets accelerate openpyxl mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 40.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 879.5/879.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.

In [5]:
from pathlib import Path
from contextlib import nullcontext
import json
import shutil
import warnings

import numpy as np
import pandas as pd
import mlflow

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)
from bert_family_utils import (
    resolve_train_test_paths,
    make_hf_dataset,
    compute_metrics_binary,
    evaluate_trainer_on_dataset,
    trainer_history_to_dataframe,
    make_jsonable_config,
)

seed_everything(42)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)
warnings.filterwarnings("ignore")
print("MLflow :", mlflow.__version__)


MLflow : 3.12.0


In [7]:
MODEL_NAME = "bert-base-uncased"
PIPELINE_NAME = "BERT_base_uncased"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

BASELINE_CONFIG = {
  "learning_rate": 2e-05,
  "batch_size": 8,
  "num_epochs": 3,
  "weight_decay": 0.01,
  "max_len": 96
}

TUNING_CANDIDATES = [
  {
    "learning_rate": 2e-05,
    "batch_size": 8,
    "num_epochs": 3,
    "weight_decay": 0.01,
    "max_len": 96
  },
  {
    "learning_rate": 3e-05,
    "batch_size": 8,
    "num_epochs": 1,
    "weight_decay": 0.01,
    "max_len": 96
  },
  {
    "learning_rate": 2e-05,
    "batch_size": 16,
    "num_epochs": 1,
    "weight_decay": 0.01,
    "max_len": 96
  }
]

OUTPUT_DIR = Path("outputs/BERT_bert_base_uncased")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASELINE_ARTIFACTS_DIR = OUTPUT_DIR / "baseline_artifacts"
TUNING_ARTIFACTS_DIR = OUTPUT_DIR / "tuning_artifacts"
BASELINE_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
TUNING_ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_STEM = "BERT_bert_base_uncased"

USE_MLFLOW = True
MLFLOW_EXPERIMENT_NAME = "DT_BERT_BERTBaseUncased"
MLFLOW_TRACKING_URI = Path("outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False


In [8]:
if USE_MLFLOW:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    print("MLflow tracking URI :", MLFLOW_TRACKING_URI)
    print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)
else:
    print("MLflow désactivé.")


2026/05/11 00:23:54 INFO mlflow.tracking.fluent: Experiment with name 'DT_BERT_BERTBaseUncased' does not exist. Creating a new experiment.


MLflow tracking URI : file:///content/outputs/mlruns
MLflow experiment   : DT_BERT_BERTBaseUncased


In [9]:
TRAIN_PATH, TEST_PATH = "/content/train.csv", "/content/test.csv"

df_train_full, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Train complet :", len(X_train_full))
print("Train tuning  :", len(X_train))
print("Validation    :", len(X_val))
print("Test          :", len(X_test))


Train complet : 9096
Train tuning  : 8186
Validation    : 910
Test          : 2274


In [10]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_ds = make_hf_dataset(tokenizer, X_train, y_train, max_len=BASELINE_CONFIG["max_len"])
val_ds = make_hf_dataset(tokenizer, X_val, y_val, max_len=BASELINE_CONFIG["max_len"])
test_ds = make_hf_dataset(tokenizer, X_test, y_test, max_len=BASELINE_CONFIG["max_len"])
train_full_ds = make_hf_dataset(tokenizer, X_train_full, y_train_full, max_len=BASELINE_CONFIG["max_len"])


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8186 [00:00<?, ? examples/s]

Map:   0%|          | 0/910 [00:00<?, ? examples/s]

Map:   0%|          | 0/2274 [00:00<?, ? examples/s]

Map:   0%|          | 0/9096 [00:00<?, ? examples/s]

## Phase 1 — Modèle non tuné

Dans cette première partie, on entraîne le modèle avec une configuration de départ raisonnable. Cette étape permet de disposer d’une baseline transformer claire avant toute optimisation.


In [11]:
def make_training_args(
    output_dir: str,
    learning_rate: float,
    batch_size: int,
    num_epochs: int,
    weight_decay: float,
    do_eval: bool = True,
    load_best_model_at_end: bool = True,
):
    return TrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch" if do_eval else "no",
        save_strategy="epoch" if do_eval else "no",
        logging_strategy="epoch",
        learning_rate=learning_rate,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        num_train_epochs=num_epochs,
        weight_decay=weight_decay,
        load_best_model_at_end=load_best_model_at_end if do_eval else False,
        metric_for_best_model="f1" if do_eval else None,
        greater_is_better=True if do_eval else None,
        report_to="none",
        save_total_limit=1 if do_eval else None,
        seed=42,
    )


In [ ]:
baseline_run_ctx = mlflow.start_run(run_name=f"BASE_{PIPELINE_NAME}") if USE_MLFLOW else nullcontext()

with baseline_run_ctx:
    baseline_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    baseline_args = make_training_args(
        output_dir=str(BASELINE_ARTIFACTS_DIR / "trainer_outputs"),
        learning_rate=BASELINE_CONFIG["learning_rate"],
        batch_size=BASELINE_CONFIG["batch_size"],
        num_epochs=BASELINE_CONFIG["num_epochs"],
        weight_decay=BASELINE_CONFIG["weight_decay"],
        do_eval=True,
        load_best_model_at_end=True,
    )

    baseline_trainer = Trainer(
        model=baseline_model,
        args=baseline_args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics_binary,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
    )

    baseline_trainer.train()

    baseline_history_df = trainer_history_to_dataframe(baseline_trainer)
    if not baseline_history_df.empty:
        baseline_history_df.to_csv(BASELINE_ARTIFACTS_DIR / f"{PIPELINE_NAME}_baseline_history.csv", index=False)

    baseline_train_metrics = evaluate_trainer_on_dataset(baseline_trainer, train_ds, y_train, "train")
    baseline_test_metrics = evaluate_trainer_on_dataset(baseline_trainer, test_ds, y_test, "test")

    baseline_result = {"pipeline": PIPELINE_NAME}
    baseline_result.update(baseline_train_metrics)
    baseline_result.update(baseline_test_metrics)
    baseline_result["config"] = make_jsonable_config(BASELINE_CONFIG)

    if USE_MLFLOW:
        mlflow.set_tag("notebook", PIPELINE_NAME)
        mlflow.set_tag("family", "bert_like_models")
        mlflow.set_tag("phase", "baseline")
        mlflow.log_param("model_name", MODEL_NAME)
        for k, v in BASELINE_CONFIG.items():
            mlflow.log_param(f"baseline__{k}", v)
        for col, val in baseline_result.items():
            if col not in {"pipeline", "config"} and pd.notna(val):
                try:
                    mlflow.log_metric(col, float(val))
                except Exception:
                    pass
        if (BASELINE_ARTIFACTS_DIR / f"{PIPELINE_NAME}_baseline_history.csv").exists():
            mlflow.log_artifact(str(BASELINE_ARTIFACTS_DIR / f"{PIPELINE_NAME}_baseline_history.csv"), artifact_path="baseline_history")

baseline_df = round_results(pd.DataFrame([baseline_result]))
display(baseline_df)
display(round_results(metric_matrix_from_results(pd.DataFrame([baseline_result]))))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.322331,0.300339,0.904396,0.756250,0.715976,0.735562
2,0.207703,0.297764,0.918681,0.784431,0.775148,0.779762
3,0.106956,0.391403,0.916484,0.768786,0.786982,0.777778


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

pipeline,test
train_accuracy,0.9747
train_precision_macro,0.9639
train_recall_macro,0.9518
train_f1_macro,0.9577
train_precision_weighted,0.9745
train_recall_weighted,0.9747
train_f1_weighted,0.9745
train_precision_class_0,0.9808
train_recall_class_0,0.9883
train_f1_class_0,0.9845


pipeline,test
train_accuracy,0.9747
train_precision_macro,0.9639
train_recall_macro,0.9518
train_f1_macro,0.9577
train_precision_weighted,0.9745
train_recall_weighted,0.9747
train_f1_weighted,0.9745
train_precision_class_0,0.9808
train_recall_class_0,0.9883
train_f1_class_0,0.9845


## Phase 2 — Tuning léger

Dans cette seconde partie, on teste plusieurs configurations raisonnables. Le critère principal de sélection est le **F1 de la classe positive sur la validation**. Une fois la meilleure configuration identifiée, on réentraîne le modèle sur tout le train avant de l’évaluer sur le test.


In [ ]:
TUNING_CANDIDATES = [
  {
    "learning_rate": 2e-05,
    "batch_size": 8,
    "num_epochs": 3,
    "weight_decay": 0.01,
    "max_len": 96
  }
]

In [24]:
tuning_rows = []

for idx, cfg in enumerate(TUNING_CANDIDATES, start=1):
    print("=" * 100)
    print(f"Essai tuning {idx}/{len(TUNING_CANDIDATES)}")
    print(cfg)

    run_ctx = mlflow.start_run(run_name=f"TUNE_TRIAL_{idx}_{PIPELINE_NAME}") if USE_MLFLOW else nullcontext()

    with run_ctx:
        trial_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

        trial_train_ds = make_hf_dataset(tokenizer, X_train, y_train, max_len=cfg["max_len"])
        trial_val_ds = make_hf_dataset(tokenizer, X_val, y_val, max_len=cfg["max_len"])

        trial_args = make_training_args(
            output_dir=str(TUNING_ARTIFACTS_DIR / f"trial_{idx}_trainer_outputs"),
            learning_rate=cfg["learning_rate"],
            batch_size=cfg["batch_size"],
            num_epochs=cfg["num_epochs"],
            weight_decay=cfg["weight_decay"],
            do_eval=True,
            load_best_model_at_end=True,
        )

        trial_trainer = Trainer(
            model=trial_model,
            args=trial_args,
            train_dataset=trial_train_ds,
            eval_dataset=trial_val_ds,
            data_collator=data_collator,
            compute_metrics=compute_metrics_binary,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
        )

        trial_trainer.train()

        val_metrics = evaluate_trainer_on_dataset(trial_trainer, trial_val_ds, y_val, "val")
        row = {
            "trial_id": idx,
            "config": make_jsonable_config(cfg),
            "learning_rate": cfg["learning_rate"],
            "batch_size": cfg["batch_size"],
            "num_epochs": cfg["num_epochs"],
            "weight_decay": cfg["weight_decay"],
            "max_len": cfg["max_len"],
        }
        row.update(val_metrics)
        tuning_rows.append(row)

        history_df = trainer_history_to_dataframe(trial_trainer)
        history_path = TUNING_ARTIFACTS_DIR / f"{PIPELINE_NAME}_trial_{idx}_history.csv"
        if not history_df.empty:
            history_df.to_csv(history_path, index=False)

        if USE_MLFLOW:
            mlflow.set_tag("notebook", PIPELINE_NAME)
            mlflow.set_tag("family", "bert_like_models")
            mlflow.set_tag("phase", "tuning_trial")
            mlflow.log_param("model_name", MODEL_NAME)
            for k, v in cfg.items():
                mlflow.log_param(f"trial__{k}", v)
            for col, val in row.items():
                if col not in {"config"} and pd.notna(val):
                    try:
                        mlflow.log_metric(col, float(val))
                    except Exception:
                        pass
            if history_path.exists():
                mlflow.log_artifact(str(history_path), artifact_path="tuning_history")


Essai tuning 1/1
{'learning_rate': 2e-05, 'batch_size': 8, 'num_epochs': 3, 'weight_decay': 0.01, 'max_len': 96}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/8186 [00:00<?, ? examples/s]

Map:   0%|          | 0/910 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.315787,0.255235,0.904396,0.780822,0.674556,0.723810
2,0.186511,0.313226,0.919780,0.800000,0.757396,0.778116
3,0.095692,0.387309,0.918681,0.754011,0.834320,0.792135


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

In [25]:
tuning_df = pd.DataFrame(tuning_rows).sort_values(
    by=["val_f1_class_1", "val_recall_class_1", "val_f1_macro", "val_balanced_accuracy", "val_roc_auc"],
    ascending=False,
).reset_index(drop=True)

display(round_results(tuning_df))

best_tuning_row = tuning_df.iloc[0].copy()
best_config = json.loads(best_tuning_row["config"])

print("Meilleure configuration retenue :")
print(json.dumps(best_config, ensure_ascii=False, indent=2))


pipeline,val
trial_id,1.0000
config,"{ ""learning_rate"": 2e-05, ""batch_size"": 8, ""num_epochs"": 3, ""weight_decay"": 0.01, ""max_len"": 96 }"
learning_rate,0.0000
batch_size,8.0000
num_epochs,3.0000
weight_decay,0.0100
max_len,96.0000
val_accuracy,0.9187
val_precision_macro,0.8576
val_recall_macro,0.8861


Meilleure configuration retenue :
{
  "learning_rate": 2e-05,
  "batch_size": 8,
  "num_epochs": 3,
  "weight_decay": 0.01,
  "max_len": 96
}


## Réentraînement final avec la meilleure configuration

La meilleure configuration identifiée sur la validation est maintenant réentraînée sur tout le train disponible. C’est ce modèle réentraîné qui est ensuite évalué sur le test.


In [12]:
final_run_ctx = mlflow.start_run(run_name=f"TUNED_FINAL_{PIPELINE_NAME}") if USE_MLFLOW else nullcontext()

with final_run_ctx:
    final_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    final_train_full_ds = make_hf_dataset(tokenizer, X_train_full, y_train_full, max_len=best_config["max_len"])
    final_test_ds = make_hf_dataset(tokenizer, X_test, y_test, max_len=best_config["max_len"])

    final_args = make_training_args(
        output_dir=str(TUNING_ARTIFACTS_DIR / "best_final_trainer_outputs"),
        learning_rate=best_config["learning_rate"],
        batch_size=best_config["batch_size"],
        num_epochs=best_config["num_epochs"],
        weight_decay=best_config["weight_decay"],
        do_eval=False,
        load_best_model_at_end=False,
    )

    final_trainer = Trainer(
        model=final_model,
        args=final_args,
        train_dataset=final_train_full_ds,
        data_collator=data_collator,
        compute_metrics=compute_metrics_binary,
    )

    final_trainer.train()

    tuned_train_metrics = evaluate_trainer_on_dataset(final_trainer, final_train_full_ds, y_train_full, "train")
    tuned_test_metrics = evaluate_trainer_on_dataset(final_trainer, final_test_ds, y_test, "test")

    tuned_result = {"pipeline": PIPELINE_NAME}
    tuned_result.update(tuned_train_metrics)
    tuned_result.update(tuned_test_metrics)
    tuned_result["best_config"] = make_jsonable_config(best_config)
    tuned_result["best_val_f1_class_1"] = float(best_tuning_row["val_f1_class_1"])
    tuned_result["best_val_recall_class_1"] = float(best_tuning_row["val_recall_class_1"])
    tuned_result["best_val_f1_macro"] = float(best_tuning_row["val_f1_macro"])
    tuned_result["best_val_balanced_accuracy"] = float(best_tuning_row["val_balanced_accuracy"])
    tuned_result["best_val_roc_auc"] = float(best_tuning_row["val_roc_auc"])

    if USE_MLFLOW:
        mlflow.set_tag("notebook", PIPELINE_NAME)
        mlflow.set_tag("family", "bert_like_models")
        mlflow.set_tag("phase", "tuned_final")
        mlflow.log_param("model_name", MODEL_NAME)
        for k, v in best_config.items():
            mlflow.log_param(f"best__{k}", v)
        for col, val in tuned_result.items():
            if col not in {"pipeline", "best_config"} and pd.notna(val):
                try:
                    mlflow.log_metric(col, float(val))
                except Exception:
                    pass


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/9096 [00:00<?, ? examples/s]

Map:   0%|          | 0/2274 [00:00<?, ? examples/s]

Step,Training Loss
1137,0.321018
2274,0.197336


Step,Training Loss
1137,0.321018
2274,0.197336
3411,0.099030


In [14]:
tuned_df = round_results(pd.DataFrame([tuned_result]))
display(tuned_df)
display(round_results(metric_matrix_from_results(pd.DataFrame([tuned_result]))))


pipeline,test
train_accuracy,0.9875
train_precision_macro,0.9770
train_recall_macro,0.9818
train_f1_macro,0.9794
train_precision_weighted,0.9875
train_recall_weighted,0.9875
train_f1_weighted,0.9875
train_precision_class_0,0.9938
train_recall_class_0,0.9908
train_f1_class_0,0.9923


pipeline,test
train_accuracy,0.9875
train_precision_macro,0.9770
train_recall_macro,0.9818
train_f1_macro,0.9794
train_precision_weighted,0.9875
train_recall_weighted,0.9875
train_f1_weighted,0.9875
train_precision_class_0,0.9938
train_recall_class_0,0.9908
train_f1_class_0,0.9923


## Comparaison baseline vs tuned

Le tableau suivant permet de comparer rapidement la version non tunée et la version tunée du modèle.


In [15]:
comparison_df = pd.DataFrame([
    {
        "pipeline": PIPELINE_NAME,
        "baseline_test_f1_class_1": baseline_result["test_f1_class_1"],
        "tuned_test_f1_class_1": tuned_result["test_f1_class_1"],
        "delta_test_f1_class_1": tuned_result["test_f1_class_1"] - baseline_result["test_f1_class_1"],
        "baseline_test_recall_class_1": baseline_result["test_recall_class_1"],
        "tuned_test_recall_class_1": tuned_result["test_recall_class_1"],
        "delta_test_recall_class_1": tuned_result["test_recall_class_1"] - baseline_result["test_recall_class_1"],
        "baseline_test_f1_macro": baseline_result["test_f1_macro"],
        "tuned_test_f1_macro": tuned_result["test_f1_macro"],
        "delta_test_f1_macro": tuned_result["test_f1_macro"] - baseline_result["test_f1_macro"],
        "baseline_test_balanced_accuracy": baseline_result["test_balanced_accuracy"],
        "tuned_test_balanced_accuracy": tuned_result["test_balanced_accuracy"],
        "delta_test_balanced_accuracy": tuned_result["test_balanced_accuracy"] - baseline_result["test_balanced_accuracy"],
    }
])

display(round_results(comparison_df))


pipeline,BERT_base_uncased
baseline_test_f1_class_1,0.7715
tuned_test_f1_class_1,0.7803
delta_test_f1_class_1,0.0088
baseline_test_recall_class_1,0.7943
tuned_test_recall_class_1,0.8227
delta_test_recall_class_1,0.0284
baseline_test_f1_macro,0.8587
tuned_test_f1_macro,0.8633
delta_test_f1_macro,0.0046
baseline_test_balanced_accuracy,0.8669


## Exports

Les résultats baseline, tuning et comparaison sont exportés dans le dossier de sortie du modèle.


In [16]:
baseline_export_path = OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_results.csv"
tuning_export_path = OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_validation_results.csv"
tuned_export_path = OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_results.csv"
comparison_export_path = OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv"

pd.DataFrame([baseline_result]).to_csv(baseline_export_path, index=False)
pd.DataFrame(tuning_rows).to_csv(tuning_export_path, index=False)
pd.DataFrame([tuned_result]).to_csv(tuned_export_path, index=False)
comparison_df.to_csv(comparison_export_path, index=False)

print("Export baseline  :", baseline_export_path)
print("Export tuning    :", tuning_export_path)
print("Export tuned     :", tuned_export_path)
print("Export comparaison :", comparison_export_path)


Export baseline  : outputs/BERT_bert_base_uncased/BERT_bert_base_uncased_baseline_results.csv
Export tuning    : outputs/BERT_bert_base_uncased/BERT_bert_base_uncased_tuning_validation_results.csv
Export tuned     : outputs/BERT_bert_base_uncased/BERT_bert_base_uncased_tuned_results.csv
Export comparaison : outputs/BERT_bert_base_uncased/BERT_bert_base_uncased_baseline_vs_tuned.csv


In [17]:
from google.colab import files
import shutil

# dossier à exporter
folder_path = "/content/outputs"

# nom du zip final
output_zip = "/content/outputs"

# création du fichier zip
shutil.make_archive(output_zip, 'zip', folder_path)

# téléchargement automatique
files.download(output_zip + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>